In [ ]:
# TV OFF / ACE-Step 1.5 — production Original Music generator# Kaggle's current base image is Python 3.13; upstream ACE-Step currently# requires Python <3.13. We therefore create an isolated Python 3.12 venv.import os, sys, json, subprocess, shutil, secretsfrom pathlib import PathWORK = Path("/kaggle/working")REPO = Path("/tmp/ACE-Step-1.5")OUT = WORK / "ace_step_output"OUT.mkdir(parents=True, exist_ok=True)def run(cmd, cwd=None):    print("+", " ".join(map(str, cmd)), flush=True)    child_env = os.environ.copy()    # Kaggle's Jupyter process exports an inline Matplotlib backend that is    # unavailable in the isolated Python 3.12 venv. Use a headless backend.    child_env["MPLBACKEND"] = "Agg"    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=child_env, check=True)# Confirm that Kaggle has the expected CUDA hardware before installing packages.import torchprint("Kaggle Python:", sys.version)print("Kaggle PyTorch:", torch.__version__)print("CUDA available:", torch.cuda.is_available())if not torch.cuda.is_available():    raise RuntimeError("CUDA is unavailable; refusing CPU fallback.")gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]for i, name in enumerate(gpu_names):    print("GPU", i, name, round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), "GiB")if not any("T4" in name for name in gpu_names):    raise RuntimeError(f"Expected Tesla T4; detected {gpu_names}")# Pin to the actual upstream latest release tag (v0.1.8), not a nonexistent v1.5.2 tag.if not REPO.exists():    run(["git", "clone", "--depth", "1", "--branch", "v0.1.8",         "https://github.com/ace-step/ACE-Step-1.5.git", REPO])# The upstream v0.1.8 release hardcodes float16 on pre-Ampere CUDA GPUs.# Its NaN error hint mentions ACESTEP_DTYPE, but this release does not read# that variable. Apply a guarded local patch to enable float32 for the T4.orchestrator = REPO / "acestep/core/generation/handler/init_service_orchestrator.py"orchestrator_text = orchestrator.read_text(encoding="utf-8")old_dtype_block = '''            elif resolved_device == "cuda":                if gpu_config.cuda_supports_bfloat16():                    self.dtype = torch.bfloat16                else:                    self.dtype = torch.float16                    logger.info(                        "[initialize_service] Pre-Ampere CUDA detected: "                        "using float16 instead of bfloat16."                    )'''new_dtype_block = '''            elif resolved_device == "cuda":                forced_dtype = os.environ.get("ACESTEP_DTYPE", "").strip().lower()                if forced_dtype == "float32":                    self.dtype = torch.float32                    logger.info("[initialize_service] ACESTEP_DTYPE=float32 override applied.")                elif gpu_config.cuda_supports_bfloat16():                    self.dtype = torch.bfloat16                else:                    self.dtype = torch.float16                    logger.info(                        "[initialize_service] Pre-Ampere CUDA detected: "                        "using float16 instead of bfloat16."                    )'''if old_dtype_block not in orchestrator_text:    raise RuntimeError("Expected v0.1.8 CUDA dtype block not found; refusing an unsafe source patch.")orchestrator.write_text(orchestrator_text.replace(old_dtype_block, new_dtype_block, 1), encoding="utf-8")os.environ["ACESTEP_DTYPE"] = "float32"print("Applied guarded ACE-Step T4 float32 override:", orchestrator)# ACE-Step declares Python >=3.11,<3.13. Kaggle's host is Python 3.13,# so use an isolated 3.12 runtime instead of patching upstream metadata.uv_path = shutil.which("uv")
if not uv_path:
    uv_install_dir = WORK / ".local" / "bin"
    uv_install_dir.mkdir(parents=True, exist_ok=True)
    run(["bash", "-lc", f"curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR={uv_install_dir} sh"])
    uv_path = str(uv_install_dir / "uv")
uv = Path(uv_path)
if not uv.is_file():
    raise RuntimeError(f"uv executable was not installed; expected {uv}. PATH={os.environ.get('PATH')}")
run([uv, "python", "install", "3.12"])venv = REPO / ".venv"run([uv, "venv", "--python", "3.12", str(venv)], cwd=REPO)python = venv / "bin" / "python"run([uv, "pip", "install", "--python", str(python), "-e", "."], cwd=REPO)# Generate using the isolated supported Python runtime.generation_script = r'''import json, math, shutil, hashlib, secrets, randomfrom pathlib import Pathimport numpy as npimport soundfile as sfimport torchfrom acestep.handler import AceStepHandlerfrom acestep.inference import GenerationParams, GenerationConfig, generate_musicOUT = Path("/kaggle/working/ace_step_output")seed = secrets.randbelow(2**31 - 1) + 1track_duration = random.choice([156, 168, 174, 186, 198, 204])track_number = 11 + seed % 989# Choose exactly one channel direction at random for every release.GENRES = {    "Lo-fi": {        "title": "Windowlight Reverie",        "bpm": random.choice([72, 76, 80, 84, 88]),        "caption": "Create a fully developed original instrumental lo-fi hip-hop composition for focused work and late-night listening. Warm dusty boom-bap drums with a relaxed pocket, soft swung hi-hats, deep rounded bass, mellow Rhodes and electric piano chords, intimate jazz-influenced chord extensions, delicate guitar or muted keyboard motifs, subtle tape saturation and restrained vinyl texture. Establish a memorable melodic theme, then develop it through changing chord voicings, countermelodies, drum variations, fills, breakdowns and a satisfying closing section. Rich musical phrasing and intentional arrangement throughout; organic human feel, warm and detailed mix. Instrumental only, no vocals, no speech, no long single-note drones, no static two-bar loop, no nature sounds."    },    "Jazz": {        "title": "Blue Hour Conversation",        "bpm": random.choice([82, 88, 94, 100, 108]),        "caption": "Create a sophisticated original instrumental modern jazz composition with a memorable piano-led theme, warm upright acoustic bass, expressive brushed jazz drums and tasteful tenor saxophone responses. Use rich extended harmony, ii-V-I colors, tasteful substitutions, melodic voice leading, syncopation and natural swing. Build a real ensemble arrangement: introduce the theme, develop it with a contrasting bridge, allow a short expressive piano or saxophone solo, bring the ensemble back for a varied reprise, and finish with a deliberate musical ending. Distinct melodic phrases, harmonic movement and interplay between instruments; intimate late-night club atmosphere, polished acoustic sound. Instrumental only, no vocals, no speech, no static chord loop, no synthetic drone."    },    "Electro Instrumental": {        "title": "Neon Afterglow",        "bpm": random.choice([96, 102, 108, 114, 120]),        "caption": "Create an original melodic electro instrumental track blending sophisticated downtempo electronica and warm analog synth textures. Use a defined kick and snare groove, crisp syncopated percussion, rounded electronic bass, evolving analog arpeggios, lush polyphonic synthesizer chords, a distinctive lead melody and subtle atmospheric layers. Develop a complete arrangement with an opening motif, rhythmic entrance, evolving main theme, contrasting breakdown, energetic return and intentional outro. Use changing harmony, countermelodies, rhythmic variation and expressive filter movement; musical, memorable and emotionally engaging rather than a repetitive loop. Clean, wide, professional electronic production. Instrumental only, no vocals, no speech, no single-note stretching, no generic four-bar loop."    }}genre_label = random.choice(list(GENRES))genre = GENRES[genre_label]track_bpm = genre["bpm"]TITLE_OPTIONS = {    "Lo-fi": [        "Windowlight Reverie", "Midnight on the Balcony", "Coffee Before Sunrise",        "Soft Focus Memories", "The Quiet Side Street", "Letters in the Rain",        "Golden Hour Drift", "A Room Full of Sunday", "Velvet Afternoon",        "After the Last Train", "Blue Notes in the Dust", "Somewhere, Slowly"    ],    "Jazz": [        "Blue Hour Conversation", "The Last Table by the Window", "Velvet in the City",        "A Little After Midnight", "Smoke and Satin", "The Long Way Home",        "Autumn on 8th Street", "One More for the Road", "Moonlit Sidewalk",        "A Quiet Kind of Swing", "Between Two Heartbeats", "The Art of Staying"    ],    "Electro Instrumental": [        "Neon Afterglow", "Signals in the Dark", "Electric Daydream",        "Beyond the Skyline", "After the Blue Lights", "Parallel Horizons",        "Midnight Transmission", "A Thousand Quiet Stars", "Glass City Dreams",        "The Shape of Tomorrow", "Fading Into Motion", "Luminous Distance"    ]}track_title = random.choice(TITLE_OPTIONS[genre_label])if any(char.isdigit() for char in track_title):    raise RuntimeError("Public track title must not contain digits.")genre_prompt = genre["caption"] + f" Tempo: {track_bpm} BPM. Create a coherent complete composition of approximately {track_duration} seconds."dit = AceStepHandler()message, initialized = dit.initialize_service(    project_root="/tmp/ACE-Step-1.5",    config_path="acestep-v15-turbo",    device="cuda",    offload_to_cpu=True,    quantization="int8_weight_only",)print("ACE-Step initialization:", message, flush=True)if not initialized:    raise RuntimeError("ACE-Step DiT initialization failed.")params = GenerationParams(    task_type="text2music",    caption=genre_prompt,    lyrics="",  # Empty lyrics are required for stable instrumental generation on T4.    instrumental=True,    vocal_language="unknown",    bpm=track_bpm,    duration=track_duration,    inference_steps=8,    shift=3.0,    seed=seed,    thinking=False,    use_cot_metas=False,    use_cot_caption=False,    use_cot_language=False,)config = GenerationConfig(    batch_size=1,    allow_lm_batch=False,    use_random_seed=False,    seeds=[seed],    audio_format="wav",)result = generate_music(dit, None, params, config, save_dir=str(OUT))if not result.success:    raise RuntimeError(f"ACE-Step generation failed: {result.error}")if not result.audios:    raise RuntimeError("ACE-Step returned success but no audio output.")source = Path(result.audios[0]["path"])target = OUT / f"TV_OFF_Original_Music_{track_number:03d}_{seed}_{genre_label.replace(' ', '_')}.wav"if not source.is_file():    raise RuntimeError(f"Audio file not found: {source}")if source.resolve() != target.resolve():    shutil.copy2(source, target)info = sf.info(str(target))audio, sr = sf.read(str(target), always_2d=True, dtype="float32")peak = float(np.max(np.abs(audio)))rms = float(np.sqrt(np.mean(np.square(audio))))duration = float(info.duration)if abs(duration - track_duration) > 1.0:    raise RuntimeError(f"Unexpected audio duration: expected {track_duration}s, got {duration:.2f}s")if sr != 48000 or info.channels != 2 or info.subtype != "PCM_16":    raise RuntimeError(f"Unexpected format: {sr} Hz, {info.channels} channels, {info.subtype}")if not np.isfinite(audio).all():    raise RuntimeError("Audio contains NaN or infinite samples.")if peak < 0.01 or peak > 1.001:    raise RuntimeError(f"Invalid peak level: {peak:.5f}")if peak >= 1.0:    raise RuntimeError(f"Clipping detected: peak={peak:.6f}")if rms < 0.01 or rms > 0.35:    raise RuntimeError(f"Audio RMS too low: {rms:.6f}")manifest = {    "status": "ok",    "provider": "ACE-Step-1.5",    "release": "v0.1.8",    "model": "acestep-v15-turbo",    "python": __import__("sys").version,    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],    "cuda": True,    "duration_seconds": round(duration, 3),    "requested_duration_seconds": track_duration,    "sample_rate": int(sr),    "channels": int(info.channels),    "subtype": info.subtype,    "format": "WAV PCM_16",    "track_number": track_number,    "peak": round(peak, 6),    "rms": round(rms, 6),    "seed": seed,    "title": track_title,     "style": genre_label,     "prompt": params.caption,    "bpm": track_bpm,    "inference_steps": 8,    "instrumental": True,    "publishing_enabled": True,    "audio_file": target.name,    "sha256": hashlib.sha256(target.read_bytes()).hexdigest(),}# Create an MP3 convenience copy if ffmpeg is available.import shutil as _shutil, subprocess as _subprocessffmpeg = _shutil.which("ffmpeg")if ffmpeg:    mp3 = OUT / f"TV_OFF_Original_Music_{track_number:03d}_{seed}_{genre_label.replace(' ', '_')}.mp3"    _subprocess.run([ffmpeg, "-y", "-i", str(target), "-codec:a", "libmp3lame", "-b:a", "320k", str(mp3)], check=True, stdout=_subprocess.DEVNULL, stderr=_subprocess.PIPE)    manifest["mp3_file"] = mp3.nameelse:    print("ffmpeg unavailable; WAV master retained, MP3 copy skipped.")(OUT / "ace_step_smoke_result.json").write_text(json.dumps(manifest, indent=2))print(json.dumps(manifest, indent=2))print("PRODUCTION MUSIC GENERATED AND VALIDATED — ready for YouTube publishing.")'''script_path = WORK / "run_ace_step_smoke.py"script_path.write_text(generation_script, encoding="utf-8")run([python, str(script_path)])print("ACE-Step production music generation finished.")

In [ ]:
# TV OFF Art Director + SDXL Turbo: run after ACE-Step has exited and released GPU memory.
import hashlib, json, re, subprocess, sys
from pathlib import Path
import torch
OUT = Path('/kaggle/working/ace_step_output')
manifest_path = OUT / 'ace_step_smoke_result.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('status') != 'ok' or manifest.get('publishing_enabled') is not True:
    raise RuntimeError('ACE-Step output is not a verified production generation result')
audio_path = OUT / manifest['audio_file']
audio_hash = hashlib.sha256(audio_path.read_bytes()).hexdigest()
if audio_hash != manifest.get('sha256'):
    raise RuntimeError('Audio hash mismatch before visual generation')
if not torch.cuda.is_available(): raise RuntimeError('CUDA required for SDXL Turbo')
print('GPU inventory:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'diffusers>=0.32.0', 'transformers>=4.46.0', 'accelerate>=1.1.0', 'safetensors>=0.4.5', 'Pillow>=10.0'], check=True)
from diffusers import AutoPipelineForText2Image
from PIL import Image, ImageStat
import gc
torch.cuda.empty_cache(); gc.collect()
style = str(manifest['style']); title = str(manifest['title'])
style_l = style.lower()
if any(k in style_l for k in ['lo-fi','lofi','hip-hop']):
    family = 'cozy creative listening studio with warm timber, soft amber lamps, analog audio equipment, books and tactile textiles'
elif any(k in style_l for k in ['jazz','latin','afro','cuban']):
    family = 'warm cinematic listening room with walnut, brass, amber practical lighting and tactile acoustic details'
elif any(k in style_l for k in ['gugak','korean','japanese','indian','world','traditional']):
    family = 'refined contemporary cultural listening pavilion with natural timber, handmade paper, stone and restrained heritage details'
elif any(k in style_l for k in ['electronic','ambient','synth','downtempo']):
    family = 'minimalist futuristic hi-fi listening suite with sculptural forms, dark mineral surfaces and subtle indirect lighting'
else:
    family = 'premium contemporary music listening lounge with bespoke furniture, natural materials and cinematic architectural lighting'
seed = int(manifest.get('seed', 0)) ^ int(audio_hash[:8], 16)
generator = torch.Generator(device='cuda').manual_seed(seed)
prompt = ('Photorealistic luxury architectural interior photograph, vertical composition, '+family+'. '
          'A distinctive one-of-a-kind room designed around the mood of '+title+'. '
          'Wide static architectural camera, eye-level, balanced composition, realistic materials, physically plausible lighting, '
          'fine detail, premium editorial interior photography. Empty room, no people, no text, no typography, no letters, '
          'no logos, no watermark, no collage, no split panels, no camera movement.')
pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16', use_safetensors=True)
pipe.to('cuda')
try: pipe.enable_attention_slicing()
except Exception: pass
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, width=768, height=1360, generator=generator).images[0]
image = image.convert('RGB').resize((1080,1920), Image.Resampling.LANCZOS)
# Generate a separate native landscape artwork for the full-length upload.
# Do not stretch or place the vertical Short artwork inside a blurred canvas.
landscape_prompt = ('Photorealistic luxury architectural interior photograph, wide horizontal 16:9 composition, '+family+'. '
          'A distinctive one-of-a-kind room designed around the mood of '+title+'. '
          'Wide static architectural camera, eye-level, balanced panoramic composition with a broad view of the room, '
          'realistic materials, physically plausible lighting, fine detail, premium editorial interior photography. '
          'Empty room, no people, no text, no typography, no letters, no logos, no watermark, no collage, no split panels.')
landscape_generator = torch.Generator(device='cuda').manual_seed(seed ^ 0x5A17C3)
landscape = pipe(prompt=landscape_prompt, num_inference_steps=4, guidance_scale=0.0,
                 width=1360, height=768, generator=landscape_generator).images[0]
landscape = landscape.convert('RGB').resize((1920,1080), Image.Resampling.LANCZOS)
landscape_path = OUT / ('TV_OFF_Landscape_' + re.sub(r'[^A-Za-z0-9]+','_',title)[:48] + '.jpg')
landscape.save(landscape_path, 'JPEG', quality=95, optimize=True)
landscape_hash = hashlib.sha256(landscape_path.read_bytes()).hexdigest()
if landscape_path.stat().st_size < 100000:
    raise RuntimeError('Landscape visual failed file-size quality gate')
visual_path = OUT / ('TV_OFF_Visual_' + re.sub(r'[^A-Za-z0-9]+','_',title)[:48] + '.jpg')
image.save(visual_path, 'JPEG', quality=95, optimize=True)
stat = ImageStat.Stat(image)
if visual_path.stat().st_size < 100000 or max(stat.stddev) < 8:
    raise RuntimeError('Visual failed blank/low-detail quality gate')
visual_hash = hashlib.sha256(visual_path.read_bytes()).hexdigest()
manifest.update({'visual_file':visual_path.name,'visual_sha256':visual_hash,'landscape_file':landscape_path.name,'landscape_sha256':landscape_hash,'landscape_width':1920,'landscape_height':1080,'visual_style':style,'visual_seed':manifest.get('seed'),
                 'visual_model':'stabilityai/sdxl-turbo','visual_model_revision':'main','visual_prompt':prompt,
                 'visual_generation_seed':seed,'visual_width':1080,'visual_height':1920,'visual_format':'JPEG',
                 'pipeline_version':'tv-off-unified-sdxl-v1'})
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({k:manifest[k] for k in ['title','style','visual_file','visual_sha256','visual_model','visual_generation_seed','visual_width','visual_height']},indent=2))
print('UNIFIED GENERATION PASSED — original music + unique SDXL visual; publishing enabled.')
